In [1]:
# | echo: false

import os
import re
from pathlib import Path
from typing import Dict, List, Set, Optional

import duckdb
import numpy as np
import pandas as pd
from dotenv import load_dotenv

In [2]:
print(f"DuckDB version: {duckdb.__version__}")

DuckDB version: 1.1.2


In [3]:
def get_schema_info(conn):
    # Get table information
    tables_query = """
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_schema != 'information_schema'
    ORDER BY table_schema, table_name
    """
    tables = conn.execute(tables_query).fetchall()

    # Get column information for each table
    schema_info = {}
    for schema, table, table_type in tables:
        columns_query = f"""
        SELECT column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_schema = '{schema}' AND table_name = '{table}'
        ORDER BY ordinal_position
        """
        columns = conn.execute(columns_query).fetchall()
        
        if schema not in schema_info:
            schema_info[schema] = {}
        schema_info[schema][table] = {
            'type': table_type,
            'columns': [
                {
                    'name': col[0],
                    'type': col[1],
                    'nullable': col[2] == 'YES'
                } for col in columns
            ]
        }

    return schema_info

In [4]:
# | echo: false

CB_DATABASE = Path("../data/Experiments_CB_2023_Q4.ddb")
CB_DATABASE_MD = str(CB_DATABASE.stem)

In [5]:
# Connect to your DuckDB database
conn = duckdb.connect(str(CB_DATABASE))

# Get the schema information
schema_info = get_schema_info(conn)

# Print the schema information
for schema, tables in schema_info.items():
    print(f"Schema: {schema}")
    for table, info in tables.items():
        print(f"  Table: {table} ({info['type']})")
        for column in info['columns']:
            nullable = "NULL" if column['nullable'] else "NOT NULL"
            print(f"    {column['name']} {column['type']} {nullable}")
    print()

Schema: main
  Table: CorrectionWithExistingValues (VIEW)
    Cohort VARCHAR NULL
    ExperimentDate DATE NULL
    SessionType VARCHAR NULL
    SubjectName VARCHAR NULL
    TrialNumber INTEGER NULL
    TrialType VARCHAR NULL
    ResponseTime DOUBLE NULL
    ResponseType VARCHAR NULL
    Comment VARCHAR NULL
    LogFilename VARCHAR NULL
    CorrectTable VARCHAR NULL
    CorrectField VARCHAR NULL
    CorrectValue VARCHAR NULL
    TableToCorrect VARCHAR NULL
    FieldToCorrect VARCHAR NULL
    ValueToCorrect VARCHAR NULL
    ExistingResponseType VARCHAR NULL
    ExistingResponseTime DOUBLE NULL
    ComparisonResult VARCHAR NULL
  Table: Corrections (BASE TABLE)
    Cohort VARCHAR NULL
    ExperimentDate DATE NULL
    SessionType VARCHAR NULL
    SubjectName VARCHAR NULL
    TrialNumber INTEGER NULL
    TrialType VARCHAR NULL
    ResponseTime DOUBLE NULL
    ResponseType VARCHAR NULL
    Comment VARCHAR NULL
    LogFilename VARCHAR NULL
    CorrectTable VARCHAR NULL
    CorrectField VARCHA

In [6]:
def topological_sort(graph: Dict[str, Set[str]]) -> List[str]:
    """
    Perform a topological sort on the dependency graph.

    This implementation uses depth-first search to produce a topological ordering.
    The result is in the desired order (tables with no dependencies first).

    Args:
        graph (Dict[str, Set[str]]): The dependency graph where keys are table names
                                     and values are sets of tables they depend on.

    Returns:
        List[str]: A list of table names in topological order.
    """
    visited: Set[str] = set()
    result: List[str] = []

    def dfs(node: str) -> None:
        if node in visited:
            return
        visited.add(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor)
        result.append(node)

    for node in graph:
        if node not in visited:
            dfs(node)

    return result[::-1]  # Reverse the result to get tables with no dependencies first

In [7]:
def infer_table_order(conn):
    # Get table and foreign key information
    fk_query = """
    SELECT tc.table_schema, tc.table_name, 
           kcu.column_name, 
           ccu.table_schema AS foreign_table_schema,
           ccu.table_name AS foreign_table_name,
           ccu.column_name AS foreign_column_name 
    FROM information_schema.table_constraints AS tc 
    JOIN information_schema.key_column_usage AS kcu
      ON tc.constraint_name = kcu.constraint_name
      AND tc.table_schema = kcu.table_schema
    JOIN information_schema.constraint_column_usage AS ccu
      ON ccu.constraint_name = tc.constraint_name
      AND ccu.table_schema = tc.table_schema
    WHERE tc.constraint_type = 'FOREIGN KEY'
    """
    foreign_keys = conn.execute(fk_query).fetchall()

    # Build dependency graph
    dependencies = {}
    for schema, table, _, _, foreign_table, _ in foreign_keys:
        full_table = f"{schema}.{table}"
        full_foreign_table = f"{schema}.{foreign_table}"
        if full_table not in dependencies:
            dependencies[full_table] = set()
        dependencies[full_table].add(full_foreign_table)

    # Add tables with no foreign keys
    tables_query = """
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema != 'information_schema'
    """
    all_tables = conn.execute(tables_query).fetchall()
    for schema, table in all_tables:
        full_table = f"{schema}.{table}"
        if full_table not in dependencies:
            dependencies[full_table] = set()

    # Perform topological sort
    sorted_tables = topological_sort(dependencies)

    return sorted_tables

In [8]:
table_order = infer_table_order(conn)

In [9]:
table_order

['main.TrialCBsCorrected',
 'main.ResponseCBsCorrected',
 'main.ExperimentCBsCorrected',
 'main.EventCBsCorrected',
 'main.CorrectionWithExistingValues',
 'main.CorrectionsSplit',
 'main.ExperimentCBs',
 'main.Corrections',
 'main.TrialCBs',
 'main.ResponseCBs',
 'main.EventCBs']

In [10]:
# Over-write table order (not inferred correctly)

table_order = \
[
    'ExperimentCBs',
    'TrialCBs',
    'ResponseCBs',
    'EventCBs',
    'Corrections',
    'CorrectionsSplit',
    'CorrectionWithExistingValues',
    'ExperimentCBsCorrected',
    'TrialCBsCorrected',
    'ResponseCBsCorrected',
    'EventCBsCorrected'
] 

In [11]:
# # | echo: false


# def infer_table_order_file_based(schema_file_path: str) -> List[str]:
#     """
#     Infer the order of table creation based on foreign key dependencies.

#     This function reads a SQL schema file, analyzes the CREATE TABLE statements
#     and their foreign key relationships, and returns a list of table names in an
#     order that ensures all dependencies are met when creating the tables.

#     The order starts with tables that have no dependencies (top-level tables) and
#     ends with tables that depend on others.

#     Args:
#         schema_file_path (str): The file path to the SQL schema file.

#     Returns:
#         List[str]: A list of table names in the order they should be created,
#                    starting with tables with no dependencies.

#     Raises:
#         FileNotFoundError: If the specified schema file cannot be found.
#         ValueError: If no valid CREATE TABLE statements are found in the file.
#     """
#     # Read the schema from the file
#     try:
#         with open(schema_file_path, "r") as file:
#             schema_sql = file.read()
#     except FileNotFoundError:
#         raise FileNotFoundError(f"Schema file not found: {schema_file_path}")

#     # Remove comments and split into CREATE TABLE statements
#     schema_sql = remove_comments(schema_sql)
#     create_statements = split_create_statements(schema_sql)

#     if not create_statements:
#         raise ValueError("No valid CREATE TABLE statements found in the schema file.")

#     # Build dependency graph
#     dependencies = build_dependency_graph(create_statements)

#     # Perform topological sort
#     sorted_tables = topological_sort(dependencies)

#     return sorted_tables  # This is already in the desired order (top-level first)


# # | echo: false


# def remove_comments(sql: str) -> str:
#     """
#     Remove SQL comments from the input string.

#     Args:
#         sql (str): The input SQL string.

#     Returns:
#         str: The SQL string with comments removed.
#     """
#     return re.sub(r"--.*$", "", sql, flags=re.MULTILINE)


# def split_create_statements(sql: str) -> List[str]:
#     """
#     Split the SQL string into individual CREATE TABLE statements.

#     Args:
#         sql (str): The input SQL string.

#     Returns:
#         List[str]: A list of CREATE TABLE statements.
#     """
#     return [stmt.strip() for stmt in re.split(r"(?i)CREATE\s+OR\s+REPLACE\s+TABLE", sql)[1:]]


# def build_dependency_graph(create_statements: List[str]) -> Dict[str, Set[str]]:
#     """
#     Build a graph of table dependencies based on foreign key relationships.

#     Args:
#         create_statements (List[str]): A list of CREATE TABLE statements.

#     Returns:
#         Dict[str, Set[str]]: A dictionary where keys are table names and values
#                              are sets of tables they depend on.
#     """
#     dependencies: Dict[str, Set[str]] = {}

#     for statement in create_statements:
#         match = re.match(r"\s*(\w+)", statement)
#         if match:
#             current_table = match.group(1)
#             dependencies[current_table] = set()

#             fk_references = re.findall(r"FOREIGN KEY.*REFERENCES\s+(\w+)", statement, re.IGNORECASE)
#             for referenced_table in fk_references:
#                 dependencies[current_table].add(referenced_table)

#     return dependencies


# def topological_sort(graph: Dict[str, Set[str]]) -> List[str]:
#     """
#     Perform a topological sort on the dependency graph.

#     This implementation uses depth-first search to produce a topological ordering.
#     The result is reversed to get the desired order (tables with no dependencies first).

#     Args:
#         graph (Dict[str, Set[str]]): The dependency graph.

#     Returns:
#         List[str]: A list of table names in topological order.
#     """
#     visited: Set[str] = set()
#     result: List[str] = []

#     def dfs(node: str) -> None:
#         if node in visited:
#             return
#         visited.add(node)
#         for neighbor in graph.get(node, []):
#             if neighbor not in visited:
#                 dfs(neighbor)
#         result.append(node)

#     for node in graph:
#         if node not in visited:
#             dfs(node)

#     return result  # No need to reverse, this is already in the desired order

In [12]:
# SCHEMA = Path("../sql/create_cb_experiment_tables_ddb.sql")

In [13]:
# # | echo: false

# try:
#     table_order = infer_table_order(str(SCHEMA))
#     print("Inferred table order (top-level first):", table_order)
# except (FileNotFoundError, ValueError) as e:
#     print(f"Error: {e}")

In [14]:
def copy_database_to_motherduck(source_db_path: str, md_db_name: str, table_order: Optional[List[str]] = None):
    conn = duckdb.connect(":memory:")
    print('duckdb.connect(":memory:")')

    load_dotenv()

    md_token = os.getenv("MD_TOKEN")
    os.environ["motherduck_token"] = md_token
    if not md_token:
        raise ValueError("MotherDuck token not found in .env file")
    else:
        print("`motherduck_token` environment variable set...")

    print("ATTACH 'md:';")
    conn.sql("ATTACH 'md:';")

    try:
        # Attach the source database
        print(f"ATTACH '{source_db_path}' AS local_source_db")
        conn.execute(f"ATTACH '{source_db_path}' AS local_source_db")

        print("USE 'local_source_db'")
        conn.execute("USE 'local_source_db'")

        # Get the list of tables and views from the source database
        tables_and_views = conn.execute("""
            SELECT table_name, table_type
            FROM information_schema.tables
            WHERE table_schema = 'main'
        """).fetchall()

        # Create a dictionary to store table types
        table_types = {name: type for name, type in tables_and_views}

        # If table_order is not provided, use all tables and views in alphabetical order
        if table_order is None:
            table_order = sorted(table_types.keys())

        # Create the new database
        print(f"CREATE OR REPLACE DATABASE {md_db_name}")
        conn.execute(f"CREATE OR REPLACE DATABASE {md_db_name}")

        # Copy tables and views in the specified order
        for table_name in table_order:
            if table_name not in table_types:
                print(f"Warning: {table_name} not found in the source database. Skipping.")
                continue

            table_type = table_types[table_name]
            if table_type == 'BASE TABLE':
                print(f"Copying table: {table_name}")
                print(f"CREATE TABLE {md_db_name}.{table_name} AS SELECT * FROM local_source_db.{table_name}")
                conn.execute(f"CREATE TABLE {md_db_name}.{table_name} AS SELECT * FROM local_source_db.{table_name}")
            elif table_type == 'VIEW':
                print(f"Copying view: {table_name}")
                print(f"SELECT sql FROM duckdb_views() WHERE view_name = '{table_name}'")
                view_definition = conn.execute(f"SELECT sql FROM duckdb_views() WHERE view_name = '{table_name}'").fetchone()[0]
                # Remove the 'CREATE VIEW' part and the view name from the definition
                view_definition = view_definition.split(' AS ', 1)[1]
                print(f"CREATE OR REPLACE VIEW {table_name} AS {view_definition}")
                conn.execute(f"CREATE OR REPLACE VIEW {table_name} AS {view_definition}")


        print(f"\nINFO: Database '{md_db_name}' created and populated successfully.")

    except Exception as e:
        print(f"ERROR: An error occurred: {str(e)}")

    finally:
        conn.close()

    return md_db_name

In [15]:
# | echo: false

# Turned off upload of database for now

md_db = copy_database_to_motherduck(CB_DATABASE, CB_DATABASE_MD, table_order)

duckdb.connect(":memory:")
`motherduck_token` environment variable set...
ATTACH 'md:';


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ATTACH '../data/Experiments_CB_2023_Q4.ddb' AS local_source_db
USE 'local_source_db'


CREATE OR REPLACE DATABASE Experiments_CB_2023_Q4


Copying table: ExperimentCBs
CREATE TABLE Experiments_CB_2023_Q4.ExperimentCBs AS SELECT * FROM local_source_db.ExperimentCBs


Copying table: TrialCBs
CREATE TABLE Experiments_CB_2023_Q4.TrialCBs AS SELECT * FROM local_source_db.TrialCBs


Copying table: ResponseCBs
CREATE TABLE Experiments_CB_2023_Q4.ResponseCBs AS SELECT * FROM local_source_db.ResponseCBs


Copying table: EventCBs
CREATE TABLE Experiments_CB_2023_Q4.EventCBs AS SELECT * FROM local_source_db.EventCBs


Copying table: Corrections
CREATE TABLE Experiments_CB_2023_Q4.Corrections AS SELECT * FROM local_source_db.Corrections


Copying view: CorrectionsSplit
SELECT sql FROM duckdb_views() WHERE view_name = 'CorrectionsSplit'
CREATE OR REPLACE VIEW CorrectionsSplit AS SELECT *, main."trim"(unnest(string_to_array(CorrectTable, ','))) AS TableToCorrect, main."trim"(unnest(string_to_array(CorrectField, ','))) AS FieldToCorrect, main."trim"(unnest(string_to_array(CorrectValue, ','))) AS ValueToCorrect FROM Corrections;
Copying view: CorrectionWithExistingValues
SELECT sql FROM duckdb_views() WHERE view_name = 'CorrectionWithExistingValues'
CREATE OR REPLACE VIEW CorrectionWithExistingValues AS SELECT sc.*, CASE  WHEN (((sc.TableToCorrect = 'TrialCBs') AND (sc.FieldToCorrect = 'ResponseType'))) THEN (t.ResponseType) ELSE NULL END AS ExistingResponseType, CASE  WHEN (((sc.TableToCorrect = 'ResponseCBs') AND (sc.FieldToCorrect = 'ResponseTime'))) THEN (r.ResponseTime) ELSE NULL END AS ExistingResponseTime, CASE  WHEN (((sc.TableToCorrect = 'TrialCBs') AND (sc.FieldToCorrect = 'ResponseType'))) THEN (CASE  WHEN ((t.Re

In [16]:
def compare_databases(local_db: str, md_db: str, threshold: float = 1e-8) -> bool:
    print(f"Comparing - LOCAL: {local_db} vs MOTHERDUCK: {md_db}")

    load_dotenv()
    md_token = os.getenv("MD_TOKEN")
    os.environ["motherduck_token"] = md_token
    if not md_token:
        raise ValueError("MotherDuck token not found in .env file")

    conn_md = duckdb.connect(f"md:{md_db}?motherduck_token={md_token}")
    conn_local = duckdb.connect(local_db)

    try:
        # Get tables and views for local database
        local_objects = conn_local.execute("""
            SELECT table_name, table_type
            FROM information_schema.tables
            WHERE table_schema = 'main'
            ORDER BY table_type, table_name
        """).fetchall()

        # Get tables and views for MotherDuck database
        md_objects = conn_md.execute("""
            SELECT table_name, table_type
            FROM information_schema.tables
            WHERE table_schema = 'main'
            ORDER BY table_type, table_name
        """).fetchall()

        # Print local database objects
        print("\nLocal database objects:")
        print("Tables:")
        for obj in local_objects:
            if obj[1] == 'BASE TABLE':
                print(f"  - {obj[0]}")
        print("Views:")
        for obj in local_objects:
            if obj[1] == 'VIEW':
                print(f"  - {obj[0]}")

        # Print MotherDuck database objects
        print("\nMotherDuck database objects:")
        print("Tables:")
        for obj in md_objects:
            if obj[1] == 'BASE TABLE':
                print(f"  - {obj[0]}")
        print("Views:")
        for obj in md_objects:
            if obj[1] == 'VIEW':
                print(f"  - {obj[0]}")

        if len(local_objects) != len(md_objects):
            print(f"\nNumber of tables/views mismatch. Local: {len(local_objects)}, MotherDuck: {len(md_objects)}")
            return False

        # Comparison logic
        for table, table_type in local_objects:
            print(f"\nComparing {table_type}: {table}")

            if table_type == 'BASE TABLE':
                if not compare_schema(conn_local, conn_md, table):
                    return False
                if not compare_row_counts(conn_local, conn_md, table):
                    return False
                if not compare_data(conn_local, conn_md, table, threshold):
                    return False
            elif table_type == 'VIEW':
                if not compare_view_definitions(conn_local, conn_md, table):
                    return False
                # Optionally, you can still compare the data in views
                if not compare_row_counts(conn_local, conn_md, table):
                    return False
                if not compare_data(conn_local, conn_md, table, threshold):
                    return False

        print("\nAll tables, views, and data match!")
        return True

    finally:
        conn_local.close()
        conn_md.close()
        
def compare_schema(conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str) -> bool:
    """
    Compare the schema of a table between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.

    Returns:
        bool: True if schemas match, False otherwise.
    """
    schema_local = conn_local.execute(f"DESCRIBE {table_name}").fetchall()
    schema_md = conn_md.execute(f"DESCRIBE {table_name}").fetchall()

    # Remove any 'generated always' columns that might be present in views
    schema_local = [col for col in schema_local if 'GENERATED ALWAYS' not in col[5]]
    schema_md = [col for col in schema_md if 'GENERATED ALWAYS' not in col[5]]

    if schema_local != schema_md:
        print(f"Schema mismatch for table/view {table_name}")
        print("Local schema:")
        print(pd.DataFrame(schema_local, columns=["Column", "Type", "Null", "Key", "Default", "Extra"]))
        print("\nMotherDuck schema:")
        print(pd.DataFrame(schema_md, columns=["Column", "Type", "Null", "Key", "Default", "Extra"]))
        return False
    return True

def compare_row_counts(conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str) -> bool:
    """
    Compare the row counts of a table between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.

    Returns:
        bool: True if row counts match, False otherwise.
    """
    count_local = conn_local.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    count_md = conn_md.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

    if count_local != count_md:
        print(f"Row count mismatch for table/view {table_name}. Local: {count_local}, MotherDuck: {count_md}")
        return False
    return True

def compare_data(conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str, threshold: float = 1e-8) -> bool:
    """
    Compare the data of a table between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.
        threshold (float): Threshold for floating point comparisons.

    Returns:
        bool: True if data matches, False otherwise.
    """
    schema_local = conn_local.execute(f"DESCRIBE {table_name}").fetchall()
    column_names = [col[0] for col in schema_local]

    order_by_clause = f"ORDER BY {', '.join(column_names)}"

    data_local = conn_local.execute(f"SELECT * FROM {table_name} {order_by_clause}").fetchall()
    data_md = conn_md.execute(f"SELECT * FROM {table_name} {order_by_clause}").fetchall()

    df_local = pd.DataFrame(data_local, columns=column_names)
    df_md = pd.DataFrame(data_md, columns=column_names)

    if not df_local.equals(df_md):
        print(f"Data mismatch for table/view {table_name}")

        for column in df_local.columns:
            if not df_local[column].equals(df_md[column]):
                print(f"\nMismatch in column: {column}")
                print(f"Local dtype: {df_local[column].dtype}, MotherDuck dtype: {df_md[column].dtype}")

                # Check for floating point differences
                if np.issubdtype(df_local[column].dtype, np.number) and np.issubdtype(df_md[column].dtype, np.number):
                    max_diff = (df_local[column] - df_md[column]).abs().max()
                    print(f"Maximum absolute difference: {max_diff}")

                    if max_diff < threshold:
                        print("Difference likely due to floating point precision")
                    else:
                        print("First few mismatching rows:")
                        mismatch = ~np.isclose(df_local[column], df_md[column], equal_nan=True)
                        print(pd.concat([df_local[mismatch], df_md[mismatch]], axis=1, keys=["Local", "MotherDuck"]).head())
                else:
                    print("First few mismatching rows:")
                    mismatch = df_local[column] != df_md[column]
                    print(pd.concat([df_local[mismatch], df_md[mismatch]], axis=1, keys=["Local", "MotherDuck"]).head())

        # Check for rows that exist in one dataset but not the other
        merge_result = pd.merge(df_local, df_md, how="outer", indicator=True)
        local_only = merge_result[merge_result["_merge"] == "left_only"]
        md_only = merge_result[merge_result["_merge"] == "right_only"]

        if not local_only.empty:
            print("\nRows in Local but not in MotherDuck:")
            print(local_only.head())

        if not md_only.empty:
            print("\nRows in MotherDuck but not in Local:")
            print(md_only.head())

        if local_only.empty and md_only.empty:
            print("\nNo rows found exclusively in either database. The mismatch might be due to data type or precision differences.")

        return False
    return True

def compare_view_definitions(conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, view_name: str) -> bool:
    """
    Compare the view definitions between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        view_name (str): Name of the view to compare.

    Returns:
        bool: True if view definitions match, False otherwise.
    """
    view_def_local = conn_local.execute(f"SELECT sql FROM duckdb_views() WHERE view_name = '{view_name}'").fetchone()[0]
    view_def_md = conn_md.execute(f"SELECT sql FROM duckdb_views() WHERE view_name = '{view_name}'").fetchone()[0]
    
    # Remove 'CREATE VIEW' part and view name from both definitions
    view_def_local = view_def_local.split(' AS ', 1)[1]
    view_def_md = view_def_md.split(' AS ', 1)[1]
    
    if view_def_local != view_def_md:
        print(f"View definition mismatch for view {view_name}")
        print("Local view definition:")
        print(view_def_local)
        print("\nMotherDuck view definition:")
        print(view_def_md)
        return False
    return True

In [17]:
# | echo: false

databases_identical = compare_databases(CB_DATABASE, CB_DATABASE_MD, threshold=1e-18)
print(f"Databases are identical: {databases_identical}")

Comparing - LOCAL: ../data/Experiments_CB_2023_Q4.ddb vs MOTHERDUCK: Experiments_CB_2023_Q4



Local database objects:
Tables:
  - Corrections
  - EventCBs
  - ExperimentCBs
  - ResponseCBs
  - TrialCBs
Views:
  - CorrectionWithExistingValues
  - CorrectionsSplit
  - EventCBsCorrected
  - ExperimentCBsCorrected
  - ResponseCBsCorrected
  - TrialCBsCorrected

MotherDuck database objects:
Tables:
  - Activity
  - Corrections
  - Demographics
  - EventCBs
  - Events
  - Events
  - ExperimentCBs
  - Experiments
  - Experiments
  - Labels
  - Patient
  - Physiology
  - ResponseCBs
  - Sleep
  - TrialCBs
  - Trials
  - Trials
  - raw_titanic
Views:
  - databases

Number of tables/views mismatch. Local: 11, MotherDuck: 19
Databases are identical: False


In [18]:
# | echo: false


def compare_table_schema(
    conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str
) -> pd.DataFrame:
    """
    Compare the schema of a table between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.

    Returns:
        pd.DataFrame: A DataFrame showing any differences in schema.
    """
    query = f"""
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_name = '{table_name}'
    ORDER BY ordinal_position
    """

    df_local = conn_local.execute(query).df()
    df_md = conn_md.execute(query).df()

    df_diff = pd.concat([df_local, df_md]).drop_duplicates(keep=False)
    return df_diff


def check_numeric_precision(
    conn: duckdb.DuckDBPyConnection,
    local_db_name: str,
    md_db_name: str,
    table_name: str,
    numeric_columns: list,
) -> dict:
    """
    Check for precision differences in numeric columns.

    Args:
        conn (duckdb.DuckDBPyConnection): Connection to the database.
        local_db_name (str): Name of the local database attachment.
        md_db_name (str): Name of the MotherDuck database attachment.
        table_name (str): Name of the table to compare.
        numeric_columns (list): List of numeric column names to check.

    Returns:
        dict: A dictionary with column names as keys and maximum differences as values.
    """
    results = {}
    for column in numeric_columns:
        query = f"""
        SELECT MAX(ABS(local.{column} - md.{column})) as max_diff
        FROM {local_db_name}.{table_name} as local
        JOIN {md_db_name}.{table_name} as md ON local.TrialID = md.TrialID
        """
        print(f"Executing query for column {column}:")
        print(query)
        max_diff = conn.execute(query).fetchone()[0]
        results[column] = max_diff
    return results


def compare_timestamps(
    conn_local: duckdb.DuckDBPyConnection,
    conn_md: duckdb.DuckDBPyConnection,
    table_name: str,
    timestamp_column: str,
) -> pd.DataFrame:
    """
    Compare timestamp handling between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.
        timestamp_column (str): Name of the timestamp column to check.

    Returns:
        pd.DataFrame: A DataFrame showing mismatching timestamps.
    """
    query = f"""
    SELECT 
        local.TrialID,
        local.{timestamp_column} as local_time,
        md.{timestamp_column} as md_time
    FROM 
        {table_name} local
    JOIN 
        md:{table_name} md ON local.TrialID = md.TrialID
    WHERE 
        local.{timestamp_column} != md.{timestamp_column}
    LIMIT 10
    """
    return conn_local.execute(query).df()


def check_null_handling(
    conn_local: duckdb.DuckDBPyConnection,
    conn_md: duckdb.DuckDBPyConnection,
    table_name: str,
    columns: list,
) -> dict:
    """
    Check for differences in NULL value handling.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.
        columns (list): List of column names to check.

    Returns:
        dict: A dictionary with column names as keys and mismatch counts as values.
    """
    results = {}
    for column in columns:
        query = f"""
        SELECT 
            COUNT(*) as mismatch_count
        FROM 
            {table_name} local
        FULL OUTER JOIN 
            md:{table_name} md ON local.TrialID = md.TrialID
        WHERE 
            (local.{column} IS NULL AND md.{column} IS NOT NULL) OR
            (local.{column} IS NOT NULL AND md.{column} IS NULL)
        """
        mismatch_count = conn_local.execute(query).fetchone()[0]
        results[column] = mismatch_count
    return results


def compare_query_plans(
    conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str
) -> tuple:
    """
    Compare query plans between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.

    Returns:
        tuple: A tuple containing the local and MotherDuck query plans.
    """
    query = f"EXPLAIN SELECT * FROM {table_name} WHERE TrialID = 1"
    local_plan = conn_local.execute(query).fetchall()
    md_plan = conn_md.execute(query).fetchall()
    return local_plan, md_plan

In [19]:
# | echo: false

# IN PROGRESS

# def compare_databases_details(local_db: str, md_db: str, table_name) -> bool:

#     print(f"Comparing - LOCAL: {local_db} vs MOTHERDUCK: {md_db}")

#     load_dotenv()
#     md_token = os.getenv("MD_TOKEN")
#     os.environ["motherduck_token"] = md_token
#     if not md_token:
#         raise ValueError("MotherDuck token not found in .env file")

#     conn_md = duckdb.connect(f"md:{md_db}?motherduck_token={md_token}")
#     conn_local = duckdb.connect(local_db)

#     # Compare table schema
#     schema_diff = compare_table_schema(conn_local, conn_md, table_name)
#     print("Schema differences:")
#     print(schema_diff)

#     # Check numeric precision

#     # Attach local database
#     conn.execute(f"ATTACH '{source_db_path}' AS local_source_db")

#     # Attach MotherDuck database

#     conn.execute(f"ATTACH 'md:{md_db_name}?motherduck_token={md_token}' AS md_db")

# # Now call the function
#     precision_diff = check_numeric_precision(conn, 'local_source_db', 'md_db', 'TrialCBs', numeric_columns)
#     numeric_columns = ['TrialNumber', 'CriterionCount']  # Add your numeric column names
#     precision_diff = check_numeric_precision(conn_local, conn_md, table_name, numeric_columns)
#     print("\nNumeric precision differences:")
#     print(precision_diff)

#     # Compare timestamps
#     timestamp_diff = compare_timestamps(conn_local, conn_md, table_name, 'TrialStartTime')
#     print("\nTimestamp differences:")
#     print(timestamp_diff)

#     # Check NULL handling
#     all_columns = ['TrialID', 'ExperimentID', 'TrialNumber', 'ResponseType', 'TrialSubType', 'CriterionType', 'CriterionCount', 'TrialStartTime', 'TrialEndTime']
#     null_diff = check_null_handling(conn_local, conn_md, table_name, all_columns)
#     print("\nNULL handling differences:")
#     print(null_diff)

#     # Compare query plans
#     local_plan, md_plan = compare_query_plans(conn_local, conn_md, table_name)
#     print("\nLocal query plan:")
#     print(local_plan)
#     print("\nMotherDuck query plan:")
#     print(md_plan)

#     conn_local.close()
#     conn_md.close()

In [20]:
# | echo: false

# Work in progress (note changed CriterionCount to an integer)

# compare_databases_details(CB_DATABASE, CB_DATABASE_MD, "TrialCBs")

### Sharing of MotherDuck copy of database

Example of accessing the MD

In [21]:
MD_SHARE = "md:_share/Experiments_CB_2023_Q4/4bb7c820-c750-460e-8ff0-f16a7bc2a1a7"

# Get the MotherDuck token from environment variable
md_token = os.getenv('MD_TOKEN')

if not md_token:
    raise ValueError("MotherDuck token not found. Please set the MD_TOKEN environment variable.")

# Connect to MotherDuck
conn = duckdb.connect('md:?motherduck_token=' + md_token)

try:
    # Attach the shared database
    conn.execute(f"ATTACH '{MD_SHARE}' AS shared_db")
    print("Shared database attached successfully.")

    # Run the COUNT query
    result = conn.execute("SELECT COUNT(*) FROM shared_db.ExperimentCBs").fetchone()
    
    if result:
        count = result[0]
        print(f"Total count of records in ExperimentCBs: {count}")
    else:
        print("No results returned from the query.")

except duckdb.Error as e:
    print(f"An error occurred: {e}")

finally:
    # Close the connection
    conn.close()

An error occurred: Catalog Error: no database/share named '_share/Experiments_CB_2023_Q4/4bb7c820-c750-460e-8ff0-f16a7bc2a1a7' found


Example (DRAFT) Java code for connecting to database share on MotherDuck - result should be 183

```java
import java.sql.*;

public class MotherDuckShareExample {
    public static void main(String[] args) {
        // JDBC URL for MotherDuck
        String jdbcUrl = "jdbc:duckdb:";

        // MotherDuck token (should be stored securely, e.g., as an environment variable)
        String mdToken = System.getenv("MD_TOKEN");

        // Share URL stored in a variable
        String MD_SHARE = "md:_share/Experiments_CB_2023_Q4/4bb7c820-c750-460e-8ff0";

        if (mdToken == null || mdToken.isEmpty()) {
            System.err.println("MotherDuck token not found. Please set the MD_TOKEN environment variable.");
            return;
        }

        try (Connection conn = DriverManager.getConnection(jdbcUrl)) {
            // Set the MotherDuck token
            try (Statement stmt = conn.createStatement()) {
                stmt.execute("SET motherduck_token='" + mdToken + "'");
            }

            // Attach the shared database
            String attachQuery = "ATTACH '" + MD_SHARE + "' AS shared_db";
            try (Statement stmt = conn.createStatement()) {
                stmt.execute(attachQuery);
                System.out.println("Shared database attached successfully.");
            }

            // Run the COUNT query
            String countQuery = "SELECT COUNT(*) FROM shared_db.ExperimentCBs";
            try (Statement stmt = conn.createStatement();
                 ResultSet rs = stmt.executeQuery(countQuery)) {
                if (rs.next()) {
                    long count = rs.getLong(1);
                    System.out.println("Total count of records in ExperimentCBs: " + count);
                }
            }

        } catch (SQLException e) {
            e.printStackTrace();
        }
    }
}
```

Similarly Javascript:

```javascript
const duckdb = require('duckdb');

// MotherDuck token (should be stored securely, e.g., as an environment variable)
const mdToken = process.env.MD_TOKEN;

// Share URL stored in a variable
const MD_SHARE = "md:_share/Experiments_CB_2023_Q4/4bb7c820-c750-460e-8ff0";

if (!mdToken) {
    console.error("MotherDuck token not found. Please set the MD_TOKEN environment variable.");
    process.exit(1);
}

// Create a new database connection
const db = new duckdb.Database(':memory:');

// Create a new connection object
const conn = new duckdb.Connection(db);

// Function to run queries
function runQuery(query) {
    return new Promise((resolve, reject) => {
        conn.all(query, (err, result) => {
            if (err) reject(err);
            else resolve(result);
        });
    });
}

async function main() {
    try {
        // Set the MotherDuck token
        await runQuery(`SET motherduck_token='${mdToken}'`);

        // Attach the shared database
        await runQuery(`ATTACH '${MD_SHARE}' AS shared_db`);
        console.log("Shared database attached successfully.");

        // Run the COUNT query
        const result = await runQuery("SELECT COUNT(*) as count FROM shared_db.ExperimentCBs");
        
        if (result && result.length > 0) {
            console.log(`Total count of records in ExperimentCBs: ${result[0].count}`);
        } else {
            console.log("No results returned from the query.");
        }
    } catch (error) {
        console.error("An error occurred:", error);
    } finally {
        // Close the connection
        conn.close();
        db.close();
    }
}

main();